In [1]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [3]:
!pip install minsearch

Defaulting to user installation because normal site-packages is not writeable


In [11]:
from elasticsearch import Elasticsearch
es_client = Elasticsearch('http://localhost:9200') 

In [12]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}
index_name = "course-questions"

es_client.indices.create(index=index_name, body=index_settings)
from tqdm.auto import tqdm #adding progress_bar
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

In [13]:
query = "How do execute a command on a Kubernetes pod?"

In [14]:
search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"], #, "section"
                    "type": "best_fields"
                }
            }
            }
        }
}
response = es_client.search(index=index_name, body = search_query)

In [5]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [8]:
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [10]:
import sys
import os
# Get the parent directory of the current notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(parent_dir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(sys.path)
from api_call import chat_with_openrouter

/home/ubuser/llm-zoomcamp
['/home/ubuser/llm-zoomcamp', '/home/ubuser/spark/spark-3.3.2-bin-hadoop3/python/lib/py4j-0.10.9.5-src.zip', '/home/ubuser/spark/spark-3.3.2-bin-hadoop3/python', '/home/ubuser/llm-zoomcamp/0a_agents_rag', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/ubuser/.local/lib/python3.10/site-packages', '/usr/local/lib/python3.10/dist-packages', '/usr/local/lib/python3.10/dist-packages/eflomal-0.1-py3.10-linux-x86_64.egg', '/usr/lib/python3/dist-packages', '/usr/lib/python3.10/dist-packages', '/home/ubuser/.local/lib/python3.10/site-packages/setuptools/_vendor']
